In [1]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("../data/olist_orders_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# Convert delivery dates
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Delivery delay
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

# Start with customer + order
journey = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days"
    ]
].merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

# Add seller and product information
item_journey = order_items[
    [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].merge(
    products[
        [
            "product_id",
            "product_category_name"
        ]
    ],
    on="product_id",
    how="left"
)

journey = journey.merge(
    item_journey,
    on="order_id",
    how="left"
)

# Add seller information
journey = journey.merge(
    sellers[
        [
            "seller_id",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

# Add customer review
journey = journey.merge(
    reviews[
        [
            "order_id",
            "review_score",
            "review_comment_message"
        ]
    ],
    on="order_id",
    how="left"
)

print("Customer journey dataset shape:", journey.shape)

print("\nJourney columns:")
print(journey.columns.tolist())

print("\nSample:")
print(journey.head())

Customer journey dataset shape: (114092, 19)

Journey columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'customer_unique_id', 'customer_city', 'customer_state', 'product_id', 'seller_id', 'price', 'freight_value', 'product_category_name', 'seller_city', 'seller_state', 'review_score', 'review_comment_message']

Sample:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp order_delivered_customer_date  \
0    delivered      2017-10-02 10:56:33        

In [2]:
# Customer Journey Stage Analysis

# 1. Customer stage
unique_customers = journey["customer_unique_id"].nunique()

# 2. Order stage
total_orders = journey["order_id"].nunique()
delivered_orders = journey.loc[
    journey["order_status"] == "delivered",
    "order_id"
].nunique()

# 3. Seller stage
unique_sellers = journey["seller_id"].nunique()

# 4. Product stage
unique_products = journey["product_id"].nunique()

# 5. Delivery stage
delivery_data = journey.drop_duplicates("order_id").copy()

delivered = delivery_data[
    delivery_data["order_delivered_customer_date"].notna()
].copy()

late_delivery_rate = (
    (delivered["delivery_delay_days"] > 0).mean() * 100
)

average_delivery_delay = delivered["delivery_delay_days"].mean()

# 6. Review stage
review_data = journey.drop_duplicates("order_id")

reviewed_orders = review_data[
    review_data["review_score"].notna()
].shape[0]

average_review_score = review_data["review_score"].mean()

negative_review_rate = (
    (review_data["review_score"] <= 3).mean() * 100
)

print("CUSTOMER JOURNEY SUMMARY")
print("-" * 40)

print("Unique customers:", unique_customers)
print("Total orders:", total_orders)
print("Delivered orders:", delivered_orders)
print("Unique sellers:", unique_sellers)
print("Unique products:", unique_products)

print("\nDELIVERY STAGE")
print("Late delivery rate:", round(late_delivery_rate, 2), "%")
print("Average delivery delay:", round(average_delivery_delay, 2), "days")

print("\nREVIEW STAGE")
print("Reviewed orders:", reviewed_orders)
print("Average review score:", round(average_review_score, 2))
print("Negative review rate:", round(negative_review_rate, 2), "%")

CUSTOMER JOURNEY SUMMARY
----------------------------------------
Unique customers: 96096
Total orders: 99441
Delivered orders: 96478
Unique sellers: 3095
Unique products: 32951

DELIVERY STAGE
Late delivery rate: 8.11 %
Average delivery delay: -11.18 days

REVIEW STAGE
Reviewed orders: 98673
Average review score: 4.09
Negative review rate: 22.74 %


In [3]:
# Identify where the customer journey breaks down

journey_analysis = pd.DataFrame({
    "Stage": [
        "Customer",
        "Order",
        "Seller",
        "Product",
        "Delivery",
        "Review"
    ],
    "Metric": [
        unique_customers,
        total_orders,
        unique_sellers,
        unique_products,
        round(late_delivery_rate, 2),
        round(negative_review_rate, 2)
    ],
    "Unit": [
        "unique customers",
        "orders",
        "sellers",
        "products",
        "late %",
        "negative review %"
    ]
})

print(journey_analysis)

      Stage    Metric               Unit
0  Customer  96096.00   unique customers
1     Order  99441.00             orders
2    Seller   3095.00            sellers
3   Product  32951.00           products
4  Delivery      8.11             late %
5    Review     22.74  negative review %


In [4]:
# Delivery → Customer Review breakdown

delivery_review = journey.drop_duplicates("order_id").copy()

delivery_review = delivery_review[
    delivery_review["delivery_delay_days"].notna()
    & delivery_review["review_score"].notna()
].copy()

delivery_review["delivery_group"] = np.select(
    [
        delivery_review["delivery_delay_days"] <= -7,
        delivery_review["delivery_delay_days"] < 0,
        delivery_review["delivery_delay_days"] == 0,
        delivery_review["delivery_delay_days"] <= 7,
        delivery_review["delivery_delay_days"] > 7
    ],
    [
        "More than 7 days early",
        "1–7 days early",
        "On time",
        "1–7 days late",
        "More than 7 days late"
    ],
    default="Unknown"
)

breakdown = (
    delivery_review
    .groupby("delivery_group")
    .agg(
        orders=("order_id", "count"),
        average_review=("review_score", "mean")
    )
    .reset_index()
)

breakdown["negative_review_rate"] = (
    delivery_review
    .groupby("delivery_group")["review_score"]
    .apply(lambda x: (x <= 3).mean() * 100)
    .values
)

breakdown = breakdown.round(2)

print("DELIVERY → REVIEW BREAKDOWN")
print("-" * 40)
print(breakdown)

DELIVERY → REVIEW BREAKDOWN
----------------------------------------
           delivery_group  orders  average_review  negative_review_rate
0          1–7 days early   17234            4.20                 20.09
1           1–7 days late    4409            3.18                 49.17
2  More than 7 days early   70934            4.32                 16.50
3   More than 7 days late    3253            1.73                 87.46
